In [ ]:
import os
import json

def process_data(json_data):
    if isinstance(json_data, dict):
        mod = 0
        for dw in json_data.get("DisplayWindowSettings"):
            for display in dw.get("DisplaySettings"):
                if "CurrentPrefSet" in display:
                    pref_set = display.get("CurrentPrefSet")
                    if 'Name' not in pref_set or pref_set['Name'][0] != '/':
                        continue
                    
                    lc = pref_set['ListConfigurations']
                    for k, v in lc.items():
                        loc = v['Location'].replace(' ', '')
                        lx, ly = loc.split(',')
                        lx = float(lx) / 2
                        ly = float(ly) / 2
                        v['Location'] = f"{int(lx)}, {int(ly)}"
                        lc[k] = v
                    pref_set['ListConfigurations'] = lc
                    mod += 1

        if mod > 0:
            json_data['Name'] = 'Mod ' + json_data['Name']
            json_data['Id'] = 'mod-' + json_data['Id']
            return json_data
    else:
        mod = 0
        for pref_set in json_data:
            if 'Name' not in pref_set or pref_set['Name'][0] != '/':
                continue

            print(pref_set['Name'])
            lc = pref_set['ListConfigurations']
            for k, v in lc.items():
                loc = v['Location'].replace(' ', '')
                lx, ly = loc.split(',')
                lx = float(lx) / 2
                ly = float(ly) / 2
                v['Location'] = f"{int(lx)}, {int(ly)}"
                lc[k] = v
            pref_set['ListConfigurations'] = lc
            mod += 1

        if mod > 0:
            return json_data
            
    return None  # Return the original data unchanged

# Get the %localappdata% path
localappdata = os.getenv("LOCALAPPDATA")

# Define target directories
directories = [
    os.path.join(localappdata, "CRC", "Profiles"),
    os.path.join(localappdata, "CRC", "PrefSets", "STARS")
]

# Process all JSON files
for dir_path in directories:
    if not os.path.exists(dir_path):
        print(f"Directory not found: {dir_path}")
        continue

    for filename in os.listdir(dir_path):
        if filename.lower().endswith(".json") and 'modified' not in filename and 'mod' not in filename:
            filepath = os.path.join(dir_path, filename)
            if os.path.isfile(filepath):
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        original_data = json.load(f)

                    # Process the data
                    modified_data = process_data(original_data)
                    if modified_data is None:
                        continue

                    # Create modified filename
                    name, ext = os.path.splitext(filename)
                    modified_filename = f"mod-{name}{ext}"
                    modified_filepath = os.path.join(dir_path, modified_filename)

                    # Write the modified data
                    with open(modified_filepath, 'w', encoding='utf-8') as f:
                        json.dump(modified_data, f, indent=2)

                    print(f"{modified_filename}")

                except Exception as e:
                    pass
                    # print(f"Error processing {filepath}: {e}")